# 1. Załadowanie zestawu danych



In [2]:
# Importowanie bibliotek
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
import csv

# Łączenie z Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Zmiana folderu roboczego
%cd /content/drive/MyDrive/bq-results-20260625-152725-1782401524079

# Załadowanie zestawu danych
df = pd.read_csv('bq-results-20260625-152725-1782401524079.csv')

Mounted at /content/drive
/content/drive/MyDrive/bq-results-20260625-152725-1782401524079


# 2. Tworzenie zmiennych oraz pętli

In [3]:
# Metryki
mymetrics = [
    {'name': 'add_payment_info / session', 'numerator': 'add_payment_info', 'denominator': 'session'},
    {'name': 'add_shipping_info / session', 'numerator': 'add_shipping_info', 'denominator': 'session'},
    {'name': 'begin_checkout / session', 'numerator': 'begin_checkout', 'denominator': 'session'},
    {'name': 'new_accounts / session', 'numerator': 'new_accounts', 'denominator': 'session'}
]

# Oznaczenie grup w eksperymentach
test_group = 1
control_group = 2

# Lista, w której zbieramy wyniki z testów
results = []

for test_number in df['test'].dropna().unique():

    # Wyciąganie danych tylko dla aktualnego testu
    df_test = df[df['test'] == test_number]

    # Interacja po liście wskaźników (mymetrics)
    for metric in mymetrics:
        num_event = metric['numerator']
        den_event = metric['denominator']

        # Pobieranie licznika
        num_test = df_test[
            (df_test['test_group'] == test_group) &
            (df_test['event_name'] == num_event)
        ]['value'].sum()

        num_control = df_test[
            (df_test['test_group'] == control_group) &
            (df_test['event_name'] == num_event)
        ]['value'].sum()

        # Pobieranie mianownika
        denom_test = df_test[
            (df_test['test_group'] == test_group) &
            (df_test['event_name'] == den_event)
        ]['value'].sum()

        denom_control = df_test[
            (df_test['test_group'] == control_group) &
            (df_test['event_name'] == den_event)
        ]['value'].sum()

        # Zabezpieczenie przed dzieleniem przez zero
        if denom_test == 0 or denom_control == 0:
            continue

        # Liczenie współczynnika konwersji
        cr_test = num_test / denom_test
        cr_control = num_control / denom_control

        # Obliczanie procentowej zmiany względem grupy kontrolnej
        metric_change = ((cr_test - cr_control) / cr_control) * 100 if cr_control > 0 else 0

        # Przygotowanie danych do testu statystycznego
        counts = np.array([num_test, num_control])
        nobs = np.array([denom_test, denom_control])

        # Test statystyczny (p-value)
        try:
            z_stat, p_value = proportions_ztest(counts, nobs)
            significant = p_value < 0.05
        except (ValueError, TypeError):
            z_stat, p_value, significant = np.nan, np.nan, False

        # Zapisywanie wyniku do listy results
        results.append({
            'test_number': test_number,
            'metric': metric['name'],
            'numerator_event': num_event,
            'denominator_event': den_event,

            'numerator_control': num_control,
            'denominator_control': denom_control,
            'conversion_rate_control': cr_control,

            'numerator_test': num_test,
            'denominator_test': denom_test,
            'conversion_rate_test': cr_test,

            'metric_change': metric_change,
            'z_stat': z_stat,
            'p_value': p_value,
            'significant': significant
        })

# Tworzenie finalnego DataFrame
final_df = pd.DataFrame(results)

# Sortowanie wyników po numerze testu oraz parametrze
final_df = final_df.sort_values(by=['test_number', 'metric'])

In [4]:
# Zapisywanie finalnego DataFrame na dysku Google
final_df.to_csv('ab_test_portfolio.csv', index=False, sep=',', quoting=csv.QUOTE_NONNUMERIC)

# [3. Dashboard Tableau](https://public.tableau.com/app/profile/bartosz.cho.odzi.ski/viz/PortfolioProject2_17912004182610/Dashboard1?publish=yes)

# 4. Podsumowanie

Test który zastosowałem to test Z dla proporcji ('proportions_ztest'). Służy do sprawdzenia, czy różnica między współczynnikami konwersji jest istotna statystycznie, czy wynika jedynie z losowego szumu. Test porównuje proporcje sukcesów do liczby prób w obu grupach, oblicza statystykę Z (wartość odchylenia od średniej) oraz wyznacza p-value. Jeśli p-value jest mniejsze od przyjętego poziomu istotności (standardowo 0.05), odrzucamy hipotezę zerową i uznajemy różnicę za istotną statystycznie.

In [5]:
# Wyświetlenie finalnej tabeli
display(final_df)

,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change,z_stat,p_value,significant
4,1,add_payment_info / session,add_payment_info,session,983,45193,0.021751,964,45362,0.021251,-2.298216,-0.518551,0.604074,False
5,1,add_shipping_info / session,add_shipping_info,session,1992,45193,0.044078,1860,45362,0.041003,-6.974377,-2.291930,0.021910,True
6,1,begin_checkout / session,begin_checkout,session,1992,45193,0.044078,1862,45362,0.041048,-6.874349,-2.258498,0.023915,True
7,1,new_accounts / session,new_accounts,session,3681,45193,0.081451,3823,45362,0.084278,3.470717,1.542883,0.122859,False
0,2,add_payment_info / session,add_payment_info,session,1125,50244,0.022391,1096,50637,0.021644,-3.333884,-0.807895,0.419151,False
1,2,add_shipping_info / session,add_shipping_info,session,2136,50244,0.042513,2194,50637,0.043328,1.918169,0.638943,0.522860,False
2,2,begin_checkout / session,begin_checkout,session,2137,50244,0.042532,2195,50637,0.043348,1.916908,0.638682,0.523030,False
3,2,new_accounts / session,new_accounts,session,4184,50244,0.083274,4165,50637,0.082252,-1.226699,-0.588793,0.556000,False
12,3,add_payment_info / session,add_payment_info,session,1788,70439,0.025384,1771,70047,0.025283,-0.396480,-0.120029,0.904461,False
13,3,add_shipping_info / session,add_shipping_info,session,2889,70439,0.041014,2915,70047,0.041615,1.464626,0.565667,0.571620,False
